In [ ]:
# ============================================================
# SKETCHBYTE — CELL 3
# SCRIPT INPUT → QWEN3-4B → TTS-READY SCRIPT (STREAMING)
# ============================================================

import gc
import torch
import os
import re
import sys
from collections import Counter
from google.colab import output
from transformers import TextIteratorStreamer
from threading import Thread

if "_sb_error" not in globals():
    def _sb_error(title, detail, hint=None):
        lines = [
            "=" * 70,
            "❌ SKETCHBYTE ERROR",
            "=" * 70,
            "",
            str(title),
        ]
        detail_text = str(detail).strip()
        if detail_text:
            lines += ["", detail_text]
        if hint:
            lines += ["", f"💡 HINT: {hint}"]
        return RuntimeError("\n".join(lines))

# Enable custom widget manager fallback
output.enable_custom_widget_manager()

print("=" * 70)
print("SKETCHBYTE — CELL 3")
print("SCRIPT INPUT + TTS FORMATTING (STREAMING)")
print("=" * 70)

# ============================================================
# 1. CHECK QWEN3-4B
# ============================================================

print("\n[1/4] Checking Qwen3-4B formatter...")
print("-" * 70)

if "formatter_model" not in globals() or formatter_model is None:
    raise _sb_error(
        "QWEN3-4B FORMATTER NOT LOADED",
        "The formatter model is missing from this session.",
        "Run Cell 2 first, then re-run this cell.",
    )

print("✅ Qwen3-4B ready.")

# ============================================================
# 2. SCRIPT INPUT (COLAB NATIVE FORM FALLBACK)
# ============================================================

print("\n[2/4] ENTER YOUR SKETCHBYTE SCRIPT")
print("-" * 70)

#@title Double-click to expand/collapse if form is hidden { display-mode: "form" }
#@markdown Enter or paste your script below, then run this cell to format it.
SCRIPT_INPUT_FORM = "" #@param {type:"string"}

# ============================================================
# FORMATTER INSTRUCTION
# ============================================================

FORMATTER_INSTRUCTION = """
You are the TTS text-preparation stage for SketchByte.
Convert a narration into clean spoken form for natural, human-sounding
text-to-speech, preserving the author's voice and vocabulary. You may apply
ONLY these transformations and nothing else:

ALLOWED (whitelist only):
1. Numbers -> words (1720 -> seventeen twenty).
2. Abbreviations/symbols -> spoken form (% -> percent, & -> and).
3. Contractions -> full forms where meaning is identical (don't -> do not,
   I've -> I have, you're -> you are, it's -> it is), and ONLY for standard
   contractions — never words like because, and, or so.
4. Punctuation-only rhythm tweaks (commas, sentence pauses) that never change
   the set of words.

FORBIDDEN:
- Adding, removing, or rewording anything outside the whitelist (no Because,
  And, So, if, also, well...).
- Rewriting grammar, 'improving' style, or paraphrasing.
- Completing a speaker's cut-off phrase: if the source stops (e.g. "But what-"),
  output it exactly; never finish the thought.
- Outputting anything besides the narration (no commentary, markers, markdown).

OUTPUT: the prepared narration only; preserve paragraphs.
"""

# ============================================================
# PROCESS SCRIPT
# ============================================================

raw_script = SCRIPT_INPUT_FORM.strip()

if not raw_script:
    raise _sb_error(
        "SCRIPT INPUT IS EMPTY",
        "The script form field is blank.",
        "Type or paste your script in the form field above, then re-run this cell.",
    )
else:
    word_count = len(raw_script.split())
    print(f"📝 Original word count: {word_count:,}")
    print("\n[3/4] Sending script to Qwen3-4B...")
    print("-" * 70)

    messages = [
        {"role": "system", "content": FORMATTER_INSTRUCTION},
        {"role": "user", "content": f"Convert to spoken TTS form using only the allowed transformations. Keep every word otherwise:\n\n{raw_script}"}
    ]

    try:
        print("🧠 Qwen3-4B formatting progress (live stream below):\n")

        # Force return as dictionary to cleanly extract input_ids
        outputs = formatter_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            enable_thinking=False
        )

        input_ids = outputs["input_ids"].to(formatter_model.device)

        # Initialize the streamer
        streamer = TextIteratorStreamer(formatter_tokenizer, skip_prompt=True, skip_special_tokens=True)

        generation_kwargs = dict(
            input_ids=input_ids,
            streamer=streamer,
            max_new_tokens=8192,
            do_sample=False,
            repetition_penalty=1.05
        )

        # Run generation in a separate thread so we can iterate over the streamer in the main thread.
        # The thread wrapper captures exceptions: a generation failure (e.g. CUDA OOM) would
        # otherwise die silently inside the thread, leaving this cell to save a partial script.
        generation_error = {}

        def _run_generation():
            try:
                formatter_model.generate(**generation_kwargs)
            except Exception as e:
                generation_error["error"] = e
                # Unblock the streamer loop in case the model failed before
                # finalizing the stream, so the main thread cannot hang.
                try:
                    streamer.end()
                except Exception:
                    pass

        thread = Thread(target=_run_generation)
        thread.start()

        # Retrieve stream and print live
        raw_formatted = ""
        for new_text in streamer:
            sys.stdout.write(new_text)
            sys.stdout.flush()
            raw_formatted += new_text

        thread.join()

        # Surface any generation error that occurred inside the thread.
        if "error" in generation_error:
            raise generation_error["error"]

        # Post-processing: remove any conversational framing the model may add
        cleaned_script = raw_formatted.strip()

        # If the model still used "---" dividers, keep only the narration parts.
        if "---" in cleaned_script:
            parts = cleaned_script.split("---")
            response_parts = [
                p for p in parts
                if p.strip() and not re.match(
                    r"^(Absolutely|Here's your|Sure,? here's?|Of course)",
                    p.strip(),
                    re.IGNORECASE,
                )
            ]
            cleaned_script = "\n".join(response_parts).strip()

        # Drop any "Sure, here is..." type framing before the real narration.
        cleaned_script = re.sub(
            r"^(Sure|Absolutely|Of course|Here's|Below is|Here is).*?\n{2,}",
            "",
            cleaned_script,
            flags=re.IGNORECASE | re.DOTALL,
        )

        # Drop any trailing commentary after the narration ends.
        cleaned_script = re.sub(
            r"\n{2,}(This version|Perfect for YouTube|Hope this helps|Let me know|Feel free).*?$",
            "",
            cleaned_script,
            flags=re.IGNORECASE | re.DOTALL,
        )

        formatted_script = cleaned_script.strip()

        # Advisory verbatim-check: warn if the formatter invented or dropped words.
        _WHITELIST_EXPAND = {
            "don't": "do not", "can't": "cannot", "won't": "will not",
            "i've": "i have", "i'm": "i am", "you're": "you are",
            "you've": "you have", "we're": "we are", "they're": "they are",
            "that's": "that is", "it's": "it is", "there's": "there is",
            "let's": "let us",
        }
        def _sfx_tokens(s):
            return re.findall(r"[a-z']+", s.lower())
        _in_flat = " ".join(_sfx_tokens(raw_script))
        _out_flat = " ".join(_sfx_tokens(formatted_script))
        for _con, _exp in _WHITELIST_EXPAND.items():
            _in_flat = _in_flat.replace(_con, _exp)
            _out_flat = _out_flat.replace(_con, _exp)
        _in_flat = _in_flat.replace("cannot", "can not")
        _out_flat = _out_flat.replace("cannot", "can not")
        _in_tok = [t for t in _in_flat.split() if not t.replace("'", "").isdigit()]
        _out_tok = [t for t in _out_flat.split() if not t.replace("'", "").isdigit()]
        _in_cnt = Counter(_in_tok)
        _out_cnt = Counter(_out_tok)
        _added = sorted((_out_cnt - _in_cnt).elements())
        _removed = sorted((_in_cnt - _out_cnt).elements())
        if _added or _removed:
            print("   ⚠️ FORMATTER WORD DRIFT (advisory):")
            if _removed:
                print(f"      removed: {_removed}")
            if _added:
                print(f"      added:   {_added}")
            print("      Review the output below; number->words may show as 'added'.")

        if not formatted_script:
            raise _sb_error(
                "FORMATTER RETURNED NO OUTPUT",
                "Qwen3-4B produced an empty result after cleaning.",
                "Re-run this cell. If it persists, restart the runtime and re-run Cells 2-3.",
            )

        formatted_words = len(formatted_script.split())

        print("\n\n" + "=" * 70)
        print("✅ FORMATTING & CLEANING COMPLETE")
        print("=" * 70)
        print(f"\nOriginal words : {word_count:,}")
        print(f"Formatted words: {formatted_words:,}")

        output_file = "/content/SketchByte_TTS_Ready_Script.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(formatted_script)

        print(f"\n💾 SCRIPT SAVED to: {output_file}")
        print("\n🟢 CELL 3 COMPLETE. Proceed to Cell 4!")

    except Exception as e:
        detail = str(e) or "Unknown error."
        hint = (
            "Reduce max_new_tokens or free GPU memory (Runtime > Restart session), "
            "then re-run Cells 2-3."
            if "out of memory" in detail.lower()
            else "Re-run this cell. If it persists, restart the runtime and re-run Cells 1-3."
        )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise _sb_error("SCRIPT FORMATTING FAILED", detail, hint) from e